# Milestone 9b - Plan and run the experiment

Part 2 of 4. In **9a** you built an eval set from production traffic: 55 labelled
questions, grouped, with a must-pass set and a dev / test / holdout split. Now
the experiment itself, through three lifecycle stages:

- **Frame**: the question, and what would count as success, written down first.
- **Design**: which versions to run, how many times, on which questions.
- **Build and trace**: run them so that every request can be looked inside later.

You won't decide anything in this notebook. Deciding is **9c**, and keeping the
two apart is part of the method.

**Do you need Ollama?** It's best with it, running `gemma3:12b`. Without it,
every model reply is replayed from a recording.

In [ ]:
import re

import ipytest
import mlflow
import pandas as pd
import pytest

from drill import rag_lab as R

ipytest.autoconfig(clean="test_*")

lab = R.Lab()
URI = R.use_mlflow("m9-help-center")
SPLIT = R.default_split()
DEV = [q for q in R.QUESTIONS if SPLIT[q["id"]] == "dev"]
print("Live: answers come from Ollama." if lab.live else "Offline: model replies are replayed from the recording.")
print(f"{len(DEV)} dev questions, {sum(1 for s in SPLIT.values() if s == 'holdout')} locked in the holdout")

## 1. Frame: write the plan before running anything

The team wants to add a **reranker**: search for the top 10 articles instead of
3, then ask the model to pick the 3 most useful. It also wants to try
**rewriting** the customer's question into help-center words before searching.
Both cost extra model calls, so time and money.

Here is the plan, written before any result exists. Every later step follows it,
and 9c applies the decision rule exactly as written.

In [ ]:
PLAN = {
    "question": "Does v2 (rewrite + search top 10 + rerank to 3) answer more questions correctly than v1 (search top 3)?",
    "change": "the retrieval steps; same model, same answer prompt, same eval set",
    "eval_set": "support-eval: the dev split for comparing, the holdout once to confirm",
    "metric": "correct = every expected fact present (or a decline, for unanswerable questions); per group and weighted by traffic",
    "guardrails": "no must-pass question that v1 passes may fail; median model time per request rises by at most 3 seconds",
    "runs": "2 per question per version, at answer temperature 0.5, to spot flaky questions",
    "test": "paired, McNemar's rule, flaky questions set aside",
    "decision_rule": "Ship v2 only if no must-pass question regresses against v1, it wins clearly by McNemar's rule "
                     "on dev, it stays within the time guardrail, and the win holds on the holdout.",
    "no_result": "Keep v1. Use the oracle and reranker checks to decide what to try next.",
}
for field, value in PLAN.items():
    print(f"{field:14} {value}")

## 2. Design: which versions, and why each one

Four versions run on the same dev questions:

| Version | What it does | Why it's in the experiment |
|---|---|---|
| **v1** | keyword search, top 3 | the baseline: what's in production |
| **v2** | rewrite, search top 10, rerank to 3 | the change being tested |
| **v2-no-rewrite** | search top 10, rerank to 3 | an **ablation**: v2 with one part removed, to see what the rewrite contributes |
| **oracle** | the right article, handed over directly | an **oracle test**: perfect retrieval, to see how much retrieval could possibly gain |

Only v1 against v2 decides anything. The other two exist to **explain** the
result. Planning them now is what lets 9c say *why* v2 won or lost, not just
whether.

**Two design checks.** First, size: how big a difference can the dev set show?
Second, how many comparisons: every extra comparison is another chance of a
fluke, which is why the winner gets confirmed on the holdout.

In [ ]:
n = len(DEV)
print(f"dev questions: {n}  ->  a pass rate is good to about ±{100 / n ** 0.5:.0f} points")
print("paired comparisons are more sensitive than that, but a small difference still won't show")

comparisons = 3   # v2 vs v1 decides; the ablation and the oracle are read against them
print(f"{comparisons} comparisons -> chance at least one wins by luck: {1 - 0.95 ** comparisons:.0%}  ->  confirm on the holdout")

## 3. Build and trace: make every request inspectable

Analysis in 9c only works if the right information is in the traces when they're
created. The app does three things for that:

- **Tags** on every trace: the version, the question id, the run, and the session.
  These let you line up two versions question by question.
- **Typed steps** (spans): the rewrite and the answer are `CHAT_MODEL`, the search is
  `RETRIEVER`, the reranker is `RERANKER`. These let you pull out one stage across
  hundreds of traces.
- **Named versions**: `mlflow.set_active_model(name="support-v2")` before running a
  version, so every trace and eval run is linked to it.

Here is one v2 request, step by step:

In [ ]:
one = lab.ask(R.BY_ID["refund-05"], "v2", run=0, tags={"stage": "demo"})
mlflow.flush_trace_async_logging()
trace = mlflow.get_trace(one["trace_id"])

print("tags:", {k: v for k, v in trace.info.tags.items() if not k.startswith("mlflow.")})
for span in trace.data.spans:
    ms = (span.end_time_ns - span.start_time_ns) / 1e6
    model = span.attributes.get("model_ms")
    took = f"{model:6} ms (model)" if model is not None else f"{ms:6.0f} ms"
    print(f"  {span.span_type:11} {span.name:10} {took}   out: {str(span.outputs)[:60]}")

**The reranker's reply needs care.** The model is asked for JSON like
`{"ids": ["kb-16", "kb-13", "kb-15"]}`. Models don't always comply: they wrap it
in words, invent an id that wasn't a candidate, or repeat one. Code that trusts
the reply breaks in production.

**Your turn.** Write `parse_ranking(reply, candidates, k)`: pull every id like
`kb-16` out of the reply, keep the ones that really are candidates, in the
model's order, without repeats; then top up from the candidates' original order;
return the first `k`.

**How to start.** `re.findall(r"kb-\d+", reply)` gives the ids in order. Walk
those, then the candidates, adding each one that's a candidate and not already
kept.

In [ ]:
def parse_ranking(reply, candidates, k):
    """The model's ranking as a list of k candidate ids, robust to messy replies."""
    # TODO: ids from the reply first (only real candidates, no repeats), then the rest of the candidates.
    raise NotImplementedError("Milestone 9b: implement parse_ranking")

In [ ]:
%%ipytest -qq

CANDIDATES = ["kb-07", "kb-08", "kb-09", "kb-16"]


def test_clean_json():
    assert parse_ranking('{"ids": ["kb-16", "kb-07", "kb-09"]}', CANDIDATES, 3) == ["kb-16", "kb-07", "kb-09"]


def test_words_around_the_json():
    assert parse_ranking('Sure! Here you go: {"ids": ["kb-09", "kb-08"]} Hope that helps', CANDIDATES, 2) == ["kb-09", "kb-08"]


def test_invented_and_repeated_ids_are_dropped():
    assert parse_ranking('["kb-99", "kb-16", "kb-16"]', CANDIDATES, 3) == ["kb-16", "kb-07", "kb-08"]


def test_unreadable_reply_keeps_the_original_order():
    assert parse_ranking("I think the second one", CANDIDATES, 2) == ["kb-07", "kb-08"]

In [ ]:
# The app uses the same logic (R.parse_ranking). Check they agree on every recorded rerank reply.
for key, reply in R.Lab(live=False).recorded.items():
    if key.startswith("rerank|"):
        cands = [f"kb-{i:02d}" for i in range(1, 26)]
        assert parse_ranking(reply, cands, 3) == R.parse_ranking(reply, cands, 3), key
print("your parse_ranking agrees with the app's on every recorded reranker reply")

## 4. Run every version, traced

Each of the four versions answers every dev question twice. Live, that's a few
hundred model calls, about 10 minutes on a laptop; offline, the recording
replays in seconds. Live results are saved for 9c.

In [ ]:
VARIANTS = ["v1", "v2", "v2-no-rewrite", "oracle"]
results = lab.run(DEV, VARIANTS, runs=2, tags={"stage": "experiment"})
saved = R.save_results(results, lab)

print(f"{len(results)} traced requests in session {lab.session}" + (f", saved to {saved}" if saved else ""))
pd.DataFrame(results).groupby(["variant", "run"])["passed"].agg(["sum", "count"])

**Resist deciding here.** You can see the raw pass counts, and it's tempting to
read a winner off them. But the plan says the decision needs the must-pass gate,
a paired test with flaky questions set aside, the time guardrail and a holdout
check, and the oracle and ablation are there to explain whatever happens.
That's **9c**.

**Browse the traces** to get a feel for them first:

```bash
mlflow ui --backend-store-uri sqlite:///.drill/m9/mlflow.db --port 5051
```

Filter by `tags.variant = 'v2'` and open a few: you'll see the rewrite, the ten
search results, the reranker's three and the answer, each with its timing.